# Jets before and after hadronization

Compare anti-$k_t$ jets from the same JEWEL event at two stages. The full `jewel-run-events` record contains the exact string inputs sent to Pythia8 in `hadronization_input` and the returned event in `hadronized_particles`. The string inputs can include beam remnants and recoil partons. The earlier cascade record is available as `particles`.

Use a notebook kernel with cppyy and install `pip install 'hepyy[hepbundle,jewel-cpp]' hepyy-utils numpy matplotlib`. Build the `jewel-run-events` executable from the [C++ checkout](../README.md#build-and-test). Set `JEWEL_BUILD_DIR` if you built outside `build-pythia8`. The Python wheel supports the hadronized final-state interface. This example uses the executable to obtain the paired records.

In [ ]:
%matplotlib inline
from collections import Counter
import hashlib
import json
import math
import os
from pathlib import Path
import subprocess
import tempfile

import matplotlib.pyplot as plt
import numpy as np

from hepyy_utils.jewel.matching import cluster_final_jets

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
build = Path(os.environ.get("JEWEL_BUILD_DIR", root / "build-pythia8")).expanduser().resolve()
runner = build / "jewel-run-events"
if not runner.is_file():
    raise FileNotFoundError(f"Build jewel-run-events or set JEWEL_BUILD_DIR (checked {runner}).")

MODES = ("vacuum", "medium")
EVENTS = int(os.environ.get("JEWEL_TARGET_EVENTS", "40"))
ATTEMPTS = EVENTS + max(1, math.ceil(0.01 * EVENTS))
GLAUBER_CSV = os.environ.get("JEWEL_GLAUBER_EVENTS")
SAMPLE_DIR = os.environ.get("JEWEL_EVENT_SAMPLE_DIR")
SEED = 2601017
RADIUS = 0.4
PT_MIN = 25.0
MAX_ABS_ETA = 2.0
MATCH_DR = 0.3

## Generate one record per event

The medium run retains recoils and applies JEWEL thermal subtraction. Set `JEWEL_GLAUBER_EVENTS` to the validated Glauber `events.csv` to sample impact parameters for 0–10% Pb–Pb with probability proportional to $N_{\rm coll}$. Without the CSV, JEWEL samples its optical centrality model. Set `JEWEL_TARGET_EVENTS=1000` for the larger study. The runner allows a small reserve of attempts and the analysis retains the first requested number of accepted events in each mode. `hadronized_particles` still contains the raw hadron record. `subtracted_particles` contains the corrected hadron momenta after subtraction using the incoming thermal references associated with retained recoils. The analysis removes entries flagged as thermal residuals and excludes neutrinos from all jet inputs. Jet-correlated recoil contributions can remain after subtraction.

The runner's validated subtraction is applied after hadronization. The vacuum raw-to-raw comparison isolates the fragmentation and decay stage for these events. In medium, raw partons vs raw hadrons keeps the same thermal treatment on both sides. The corrected-hadron result is shown alongside it to illustrate the separate effect of subtraction.

Set `JEWEL_EVENT_SAMPLE_DIR` to reuse full `vacuum.jsonl` and `medium.jsonl` outputs from the same configuration. The run is unweighted.

In [ ]:
def jet_axes(collection):
    momenta = np.asarray([p["momentum"] for p in collection
                          if abs(p["pdg"]) not in (12, 14, 16)], dtype=float).reshape(-1, 4)
    return cluster_final_jets(*momenta.T, radius=RADIUS)


def axis_distance(a, b):
    return math.hypot(a["eta"] - b["eta"], math.remainder(a["phi"] - b["phi"], math.tau))


def accepted(jet):
    return jet["pt"] >= PT_MIN and abs(jet["eta"]) < MAX_ABS_ETA


def analyze_event(event, mode, counts, samples, shifts):
    collections = {
        "Raw hadrons": (p for p in event["hadronized_particles"] if p["final"]),
    }
    if mode == "medium":
        collections["Subtracted hadrons"] = (p for p in event["subtracted_particles"]
                                             if p["final"] and not p["thermal_residual"])
    parton_jets = jet_axes(event["hadronization_input"])
    if not parton_jets:
        return
    anchor = parton_jets[0]
    matches = {}
    for label, particles in collections.items():
        jets = jet_axes(particles)
        closest = min(jets, key=lambda jet: axis_distance(anchor, jet), default=None)
        if closest is not None and axis_distance(anchor, closest) <= MATCH_DR:
            matches[label] = closest

    # Cluster and match in full phase space, then apply the parton selection.
    if not accepted(anchor):
        return
    for label in collections:
        counts["leading parton selection", label] += 1
        match = matches.get(label)
        if match is None:
            counts["unmatched", label] += 1
        elif not accepted(match):
            counts["matched outside jet cuts", label] += 1
        else:
            samples[label].append((anchor["pt"], match["pt"]))

    if mode == "medium":
        raw = matches.get("Raw hadrons")
        corrected = matches.get("Subtracted hadrons")
        if raw is None or corrected is None:
            counts["thermal pair missing axis", "Medium"] += 1
        elif axis_distance(raw, corrected) > MATCH_DR:
            counts["thermal pair axes separated", "Medium"] += 1
        else:
            shifts.append((raw["pt"], corrected["pt"], accepted(raw), accepted(corrected)))


def event_file(mode, work):
    if SAMPLE_DIR:
        return Path(SAMPLE_DIR).expanduser().resolve() / f"{mode}.jsonl"
    path = Path(work) / f"{mode}.jsonl"
    args = [str(runner), str(path), str(ATTEMPTS), str(SEED), mode,
            "--ecm", "5020", "--pt-min", "40", "--pt-max", "100", "--isr-only",
            "--table-energy", "jewel", "--longitudinal-boost", "on",
            "--jewel-fragmentation", "--keep-recoils", "on"]
    if mode == "medium":
        args += ["--centrality", "0", "10", "--subtraction-radius", str(RADIUS)]
        if GLAUBER_CSV:
            args += ["--glauber-events", str(Path(GLAUBER_CSV).expanduser().resolve())]
    run = subprocess.run(args, cwd=work, text=True, capture_output=True)
    if run.returncode:
        raise RuntimeError(run.stdout[-2000:] + run.stderr[-2000:])
    return path


def read_sample(mode, counts, samples, shifts):
    used = 0
    impact_parameters = []
    with tempfile.TemporaryDirectory(prefix="jewel-before-after-") as work:
        with event_file(mode, work).open() as stream:
            header = json.loads(next(stream))
            assert header["schema"] == "jewel-hadron-development-v1"
            assert header["mode"] == mode and header["seed"] == SEED and header["isr_only"]
            assert header["attempts_requested"] == ATTEMPTS
            assert {"PhaseSpace:pTHatMin = 40", "PhaseSpace:pTHatMax = 100"} <= set(header["pythia_settings"])
            if mode == "medium":
                assert header["subtraction"]["radius"] == RADIUS
                assert header["centrality_percent"] == [0, 10]
                if GLAUBER_CSV:
                    assert header["glauber"]["sampling"] == "ncoll-proportional"
                    assert Path(header["glauber"]["source"]).resolve() == Path(GLAUBER_CSV).resolve()
                else:
                    assert "glauber" not in header
            for line in stream:
                event = json.loads(line)
                if event["type"] == "event" and used < EVENTS:
                    assert math.isclose(event["weight"], 1)
                    analyze_event(event, mode, counts, samples, shifts)
                    used += 1
                    last_attempt = event["attempt"]
                    if mode == "medium":
                        impact_parameters.append(event["impact_parameter_fm"])
                elif event["type"] == "summary":
                    summary = event
    if used != EVENTS:
        raise RuntimeError(f"{mode}: only {used} of {EVENTS} requested accepted events were available")
    stats = dict(attempts=summary["attempts"], accepted=summary["accepted"],
                 events_analyzed=used, last_attempt_analyzed=last_attempt,
                 pythia_version=header["pythia_version"])
    if mode == "medium":
        stats["mean_b_fm"] = float(np.mean(impact_parameters))
        stats["subtraction"] = header["subtraction"]
        if GLAUBER_CSV:
            table = Path(GLAUBER_CSV)
            stats["glauber"] = {key: value for key, value in header["glauber"].items() if key != "source"}
            stats["glauber"]["csv_sha256"] = hashlib.sha256(table.read_bytes()).hexdigest()
    return stats


results, run_summary, shifts = {}, {}, []
for mode in MODES:
    labels = ["Raw hadrons"] + (["Subtracted hadrons"] if mode == "medium" else [])
    samples = {label: [] for label in labels}
    counts = Counter()
    stats = read_sample(mode, counts, samples, shifts)
    results[mode] = {label: np.asarray(sample).reshape(-1, 2)
                     for label, sample in samples.items()}
    print(f'{mode.capitalize()} — {stats["events_analyzed"]} analyzed '
          f'({stats["accepted"]}/{stats["attempts"]} generated attempts accepted)')
    stats["pairs"] = {}
    for label in labels:
        selection = dict(selected=len(samples[label]), anchors=counts["leading parton selection", label],
                         unmatched=counts["unmatched", label],
                         outside_cuts=counts["matched outside jet cuts", label])
        stats["pairs"][label] = selection
        print(f'  {label}: {selection["selected"]}/{selection["anchors"]} selected pairs, '
              f'{selection["unmatched"]} unmatched, {selection["outside_cuts"]} matched outside cuts')
    if mode == "medium":
        stats["missing_axis"] = counts["thermal pair missing axis", "Medium"]
        stats["axes_separated"] = counts["thermal pair axes separated", "Medium"]
    run_summary[mode] = stats

## Check the paired response

Each point is the leading string-input jet and its nearest jet in the specified final collection in the same event. Matching uses $Delta Rleq0.3$ in pseudorapidity and azimuth. The $p_Tgeq25$ GeV and $|eta|<2$ cuts follow matching. The medium series can contain different passing pairs because correction can change a jet axis or move it across a cut. The default is a small demonstration sample. The larger run uses the same cuts.

In [ ]:
colors = {"Raw hadrons": "#086f8f", "Subtracted hadrons": "#ae531d"}
fig, panels = plt.subplots(1, len(MODES), figsize=(6.8 * len(MODES), 5.1), squeeze=False,
                           layout="constrained")
fig.suptitle(f"Jet response on the same JEWEL events · {EVENTS:,} per sample", fontsize=17, fontweight="bold")
for mode, ax in zip(MODES, panels[0]):
    all_pt = [value.max() for value in results[mode].values() if value.size]
    upper = max(max(all_pt, default=PT_MIN + 1) * 1.06, PT_MIN + 1)
    ax.plot([PT_MIN, upper], [PT_MIN, upper], color="#708091", lw=1, ls="--", label="Equal $p_T$")
    for label, pairs in results[mode].items():
        ax.scatter(*pairs.T, s=18 if EVENTS > 100 else 37, color=colors[label], alpha=.55,
                   edgecolor="white", linewidth=.35, label=f"{label} ({len(pairs)})")
    ax.set(title="Vacuum" if mode == "vacuum" else "0–10% Pb–Pb medium",
           xlabel="String-input jet $p_T$ [GeV]", ylabel="Matched hadron jet $p_T$ [GeV]",
           xlim=(PT_MIN, upper), ylim=(PT_MIN, upper), aspect="equal")
    ax.grid(alpha=.17)
    ax.legend(frameon=False, fontsize=9)
plt.show()

## Measure the subtraction shift on the same matched jets

Find the closest raw and subtracted jets to the same leading parton anchor in full phase space and require $Delta Rleq0.3$ between the two hadron jets as well. The main shift is $p_T^{
m sub}-p_T^{
m raw}$ for these pairs after the parton cuts and before hadron-level cuts. This retains jets that subsequently cross a cut. A second result applies the hadron-level cuts to both sides. The plotted bin errors are standard errors of the paired mean.

In [ ]:
def describe(values):
    if not len(values):
        return {"count": 0}
    return {
        "count": len(values),
        "mean_gev": float(np.mean(values)),
        "se_gev": float(np.std(values, ddof=1) / np.sqrt(len(values))) if len(values) > 1 else None,
        "median_gev": float(np.median(values)),
        "fraction_negative": float(np.mean(values < 0)),
    }


def thermal_summary(points):
    raw, subtracted, raw_selected, sub_selected = np.asarray(points).T
    raw_selected, sub_selected = raw_selected.astype(bool), sub_selected.astype(bool)
    change = subtracted - raw
    common = raw_selected & sub_selected
    limits = np.array([0, 25, 35, 50, 65, 80, 100, max(110, raw.max() + 1)])
    binned = []
    for lo, hi in zip(limits, limits[1:]):
        in_bin = (raw >= lo) & (raw < hi)
        entry = describe(change[in_bin])
        if entry["count"]:
            binned.append(dict(low_gev=float(lo), high_gev=float(hi),
                               mean_raw_pt_gev=float(raw[in_bin].mean()), **entry))
    return raw, change, {
        "both_axes_matched": describe(change),
        "both_pass_hadron_cuts": describe(change[common]),
        "raw_only_passes": int(np.sum(raw_selected & ~sub_selected)),
        "subtracted_only_passes": int(np.sum(sub_selected & ~raw_selected)),
        "neither_passes": int(np.sum(~raw_selected & ~sub_selected)),
        "by_raw_pt": binned,
    }


if shifts:
    raw, change, thermal = thermal_summary(shifts)
    run_summary["thermal_shift"] = thermal
    all_pairs = thermal["both_axes_matched"]
    fig, (hist, trend) = plt.subplots(1, 2, figsize=(12, 4.5), layout="constrained")
    centrality = "Glauber-weighted" if GLAUBER_CSV else "optical"
    fig.suptitle(f"Thermal subtraction · 0–10% Pb–Pb ({centrality})", fontweight="bold", fontsize=16)
    hist.hist(change, bins="fd", color="#277b92", alpha=.86)
    hist.axvline(0, color="#506279", linestyle="--", linewidth=1, label="Zero change")
    median = all_pairs["median_gev"]
    hist.axvline(median, color="#ae531d", linewidth=1.5, label=f"Median {median:+.2f} GeV")
    hist.set(xlabel="Subtracted minus raw jet $p_T$ [GeV]", ylabel="Matched jet pairs",
             title=f'{all_pairs["count"]} stable matches before hadron cuts')
    hist.legend(frameon=False, fontsize=9)
    trend.scatter(raw, change, s=12, color="#748da0", alpha=.27, label="Individual pairs")
    for i, entry in enumerate(thermal["by_raw_pt"]):
        trend.errorbar(entry["mean_raw_pt_gev"], entry["mean_gev"], yerr=entry["se_gev"],
                       color="#9a4214", marker="o", capsize=3,
                       label="Bin mean ± standard error" if i == 0 else None)
    trend.axhline(0, color="#506279", linestyle="--", linewidth=1)
    trend.axvline(PT_MIN, color="#506279", linestyle=":", linewidth=1, label="Hadron jet $p_T$ cut")
    trend.set(xlabel="Raw hadron jet $p_T$ [GeV]", ylabel="Subtracted minus raw jet $p_T$ [GeV]",
              title="Shift versus raw jet momentum")
    trend.legend(frameon=False, fontsize=9)
    for ax in (hist, trend):
        ax.grid(axis="y", alpha=.2)
    plt.show()
else:
    print("No stable medium pairs available for a subtraction shift.")

print(json.dumps(run_summary, indent=2))